In [ ]:
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from openai import OpenAI # We still use the OpenAI library as the "driver"

# 1. Load the Baseball vs Hockey dataset
categories = ['rec.sport.baseball', 'rec.sport.hockey']



In [ ]:
newsgroups = fetch_20newsgroups(subset='train',
                                categories=categories,
                                shuffle=True, random_state=42)

In [ ]:
print(newsgroups['data'][0])

From: dougb@comm.mot.com (Doug Bank)
Subject: Re: Info needed for Cleveland tickets
Reply-To: dougb@ecs.comm.mot.com
Organization: Motorola Land Mobile Products Sector
Distribution: usa
Nntp-Posting-Host: 145.1.146.35
Lines: 17

In article <1993Apr1.234031.4950@leland.Stanford.EDU>, bohnert@leland.Stanford.EDU (matthew bohnert) writes:

|> I'm going to be in Cleveland Thursday, April 15 to Sunday, April 18.
|> Does anybody know if the Tribe will be in town on those dates, and
|> if so, who're they playing and if tickets are available?

The tribe will be in town from April 16 to the 19th.
There are ALWAYS tickets available! (Though they are playing Toronto,
and many Toronto fans make the trip to Cleveland as it is easier to
get tickets in Cleveland than in Toronto.  Either way, I seriously
doubt they will sell out until the end of the season.)

-- 
Doug Bank                       Private Systems Division
dougb@ecs.comm.mot.com          Motorola Communications Sector
dougb@nwu.edu       

In [ ]:
newsgroups['target_names']

['rec.sport.baseball', 'rec.sport.hockey']

In [ ]:
newsgroups.target_names[newsgroups['target'][0]]

'rec.sport.baseball'

In [ ]:
# 1. Total is correct
len_all = len(newsgroups.data)

# 2. Count baseball (where target index matches the name)
# In your list, 'rec.sport.baseball' is index 0
len_baseball = len([t for t in newsgroups.target if newsgroups.target_names[t] == 'rec.sport.baseball'])
len_hockey = len_all - len_baseball # The rest must be hockey

print(f"Total examples: {len_all}")
print(f"Baseball examples: {len_baseball}")
print(f"Hockey examples: {len_hockey}")

Total examples: 1197
Baseball examples: 597
Hockey examples: 600


one sameple from the baseball category can be seen above. It is an email to a mailing list. We can observe that we have 1197 ex in total, ehich are evenly splited between the 2 sports.

#Data Prepararion
We tranform the datset into a pandas dataframe, with column for promp and completion. The prompt contains the email from the mailing list, and the completion is a name of the sport, either hockey or baseball.For demontration purpose only and speed of tuning we take only 300 eg. . In a real use case the more eg the beter perfromance.

In [ ]:
# import pandas as pd
# from sklearn.datasets import fetch_20newsgroups

# # 1. Fetch the data again (to ensure fresh start)
# categories = ['rec.sport.baseball', 'rec.sport.hockey']
# newsgroups = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))

# 2. Take only 300 examples for the "Speed Test"
# We shuffle them to make sure we get a mix of both sports
df_subset = pd.DataFrame({'text': newsgroups.data, 'label': newsgroups.target})
df_subset = df_subset.sample(n=300, random_state=42).reset_index(drop=True)

# 3. Create the 'prompt' and 'completion' columns
# PROMPT: Text + Separator
df_subset['prompt'] = df_subset['text'].apply(lambda x: f"{x[:1000]} \n\n###\n\n")

# COMPLETION: Leading Space + Sport Name
df_subset['completion'] = df_subset['label'].apply(
    lambda x: f" {'baseball' if x == 0 else 'hockey'}"
)

# 4. Final look at your training data
training_df = df_subset[['prompt', 'completion']]
print(training_df.head())

                                              prompt completion
0  From: dtate+@pitt.edu (David M. Tate)\nSubject...   baseball
1  From: golchowy@alchemy.chem.utoronto.ca (Geral...     hockey
2  From: kenney@tribe.b17d.ingr.COM (David Kenney...   baseball
3  From: cmk@athena.mit.edu (Charles M Kozierok)\...   baseball
4  From: filinuk@staff.dccs.upenn.edu (Geoff Fili...     hockey


Both baseball and hockey are single tokens. we save the dataset as a jsonl file.

In [ ]:
# Assuming your DataFrame is named 'training_df' and has 'prompt' and 'completion' columns
training_df.to_json("sports_data.jsonl", orient="records", lines=True)

print("File saved as sports_data.jsonl! You're ready for training.")

File saved as sports_data.jsonl! You're ready for training.


#Data preparatn tool
we can usee a data ppreptn tool which wil sugest a few improvemtns to our dataset before fine tuning. before launching the tool we update the nvidia libry to ensure we r using the latest data prepartn tool. We additionly specify -q which auto acepts all sugestions.

In [ ]:
!pip install "nemo_toolkit[all]"
# Or for the specific curation/prep tool:
# !pip install --upgrade nemo_curator

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.6/77.6 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 8.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Us

In [ ]:
import json
import random

input_file = "sports_data.jsonl"
train_file = "sports_data_train.jsonl"
valid_file = "sports_data_valid.jsonl"

# 1. Load and clean the data
cleaned_data = []
seen_prompts = set()

with open(input_file, "r") as f:
    for line in f:
        item = json.loads(line)
        # Ensure clean formatting
        prompt = item["prompt"].strip() + " \n\n###\n\n"
        completion = " " + item["completion"].strip()

        if prompt not in seen_prompts:
            cleaned_data.append({"prompt": prompt, "completion": completion})
            seen_prompts.add(prompt)

# 2. Shuffle the data so the sports are mixed up
random.seed(42) # Keeps the result same every time you run it
random.shuffle(cleaned_data)

# 3. Calculate the Split (80% Training, 20% Validation)
split_index = int(len(cleaned_data) * 0.8)
train_data = cleaned_data[:split_index]
valid_data = cleaned_data[split_index:]

# 4. Save both files
def save_jsonl(data, filename):
    with open(filename, "w") as f:
        for entry in data:
            f.write(json.dumps(entry) + "\n")

save_jsonl(train_data, train_file)
save_jsonl(valid_data, valid_file)

print(f"✅ Success! Generated 2 files:")
print(f"   - {train_file}: {len(train_data)} examples")
print(f"   - {valid_file}: {len(valid_data)} examples")

✅ Success! Generated 2 files:
   - sports_data_train.jsonl: 240 examples
   - sports_data_valid.jsonl: 60 examples


In [ ]:
# nemo-data-prep -f sports_data.jsonl -q

SyntaxError: invalid syntax (780195504.py, line 1)

the tool helpfuly sugest a few improvemnt to dataset and split datset into training and validatn set.
a sufix btwen a prompt and a completin is necesayr to tel the model tht the input text has stoped and now it needs to prdeict the class. since we use the same operatr in each eg. , the model is able to lern it meant to precit eithe baseball or hockey folwing seprstor . A white space prefix in completion is useful , as most word token ar tokenized with a sapce prefix . the tool also recognised that this is likely a clasfication task , so it sugested, to split the dataset into training and validatn datasets. this wil alow us to easly measure expexted performance on new data.

#Fine tuning
The tool sugest we run th folowin comnd to trin the dataset . since this is a clasifictn task, we would like to know, wht the genralisatn performance on the provided validtn set is for our clasifctn use case. the tool sugest to add "compute clasification metrics --clasificatn_positve_class "" baseball" In order to compute the clasificstn metrics.

We can simply copy th sugested comnd from the cli tool. we specificaly add -m ada to fine-tune a cheaper and faster ada model , which usely comperable in performance to slower and more expensive model on clasificatn use cases.

In [ ]:
%pip install trl

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig
from datasets import load_dataset

In [ ]:
# 1. Load your local NVIDIA-ready datasets
dataset = load_dataset("json", data_files={
    "train": "sports_data_train.jsonl",
    "validation": "sports_data_valid.jsonl"
})

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [ ]:
# 2. Initialize Model and Tokenizer
model_id = "nvidia/Mistral-NeMo-12B-Instruct-v1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# tokenizer.pad_token = tokenizer.eos_token # Essential for classification

# 3. Configure LoRA (The "Adapter")
# peft_config = LoraConfig(
#     r=16,
#     lora_alpha=32,
#     target_modules=["q_proj", "v_proj"],
#     task_type="CAUSAL_LM"
# )

OSError: nvidia/Mistral-NeMo-12B-Instruct-v1 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`